In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import pandas as pd
from tqdm import trange
from sklearn.metrics import roc_auc_score

import mat73

In [ ]:
my_dict = mat73.loadmat('SingleRegion_Aggression_data.mat')

In [ ]:
TrainsetMouse = my_dict['TrainsetMouse']
mbeh_all2 = my_dict['mbeh_all2']
mcond_all2 = my_dict['mcond_all2']
mouse_all2 = my_dict['mouse_all2']
mpow_all2 = my_dict['mpow_all2']
mpow_all3s = my_dict['mpow_all3s']
mtimecondbeh2 = my_dict['mtimecondbeh2']
mu_all3 = my_dict['mu_all3']
testsetMouse = my_dict['testsetMouse']


### Get index of training set

In [ ]:
N_samples = len(mouse_all2)

mice_all = []
for mouse in TrainsetMouse:
    mice_all.append(mouse[0])
for mouse in testsetMouse:
    mice_all.append(mouse[0])
mice_all = np.array(mice_all)

trainset = []
for i in range(len(TrainsetMouse)):
    trainset.append(TrainsetMouse[i][0])

train_idxs = np.zeros(N_samples)
mouse_idxs = np.zeros(N_samples)
for i in range(N_samples):
    if mouse_all2[i][0] == 'Mouse048':
        train_idxs[i] = -1
        continue
    if mouse_all2[i][0] in trainset:
        train_idxs[i] = 1
    mouse_idxs[i] = np.where(mice_all==mouse_all2[i][0])[0][0]
np.mean(train_idxs)

### Get positive and negative indexes of conditions

In [ ]:
idxs_pos = (mcond_all2==4)&(mbeh_all2==2)
idx_neg = ((mcond_all2==4)&(mbeh_all2==1))|((mcond_all2==8)&(mbeh_all2==2))|((mcond_all2==6)&(mbeh_all2==2))
selection_indices = idxs_pos|idx_neg
print(np.mean(idxs_pos))
print(np.mean(idx_neg))
print(np.mean(selection_indices))

### Create task labels

In [ ]:
y = np.zeros(N_samples)
y[idxs_pos] = 1

In [ ]:
mpower_reduced = mpow_all2[:,:,selection_indices]
y_reduced = y[selection_indices]
train_idxs_reduced = train_idxs[selection_indices]

In [ ]:
y_train = y_reduced[train_idxs_reduced==1]
y_test = y_reduced[train_idxs_reduced==0]

In [ ]:
m_idx_unique_test = np.unique(mouse_idxs[train_idxs==0])
mouse_idxs_reduced = mouse_idxs[selection_indices]
m_test = mouse_idxs_reduced[train_idxs_reduced==0]

## Now fit all of the individual models

In [ ]:
def calc_U(y_true, y_score):
    n1 = np.sum(y_true==1)
    n0 = len(y_score)-n1
    
    ## Calculate the rank for each observation
    # Get the order: The index of the score at each rank from 0 to n
    order = np.argsort(y_score)
    # Get the rank: The rank of each score at the indices from 0 to n
    rank = np.argsort(order)
    # Python starts at 0, but statistical ranks at 1, so add 1 to every rank
    rank += 1
    
    # If the rank for target observations is higher than expected for a random model,
    # then a possible reason could be that our model ranks target observations higher
    U1 = np.sum(rank[y_true == 1]) - n1*(n1+1)/2
    U0 = np.sum(rank[y_true == 0]) - n0*(n0+1)/2
    
    # Formula for the relation between AUC and the U statistic
    AUC1 = U1/ (n1*n0)
    AUC0 = U0/ (n1*n0)
    
    return U1, AUC1, U0, AUC0

In [ ]:
nFact = 4
mu = 1.0
model_list = []
test_aucs = np.zeros(11)

test_aucs_mouse = np.zeros((11,9))
test_aucs_mouse_U = np.zeros((11,9))

for i in trange(11):
    XT = np.squeeze(mpower_reduced[:,i,:])
    X = np.transpose(XT)
    X = X*10
    X[X>6] = 6
    Xtrain = X[train_idxs_reduced==1]
    Xtest = X[train_idxs_reduced==0]
    model = lm.LogisticRegressionCV(max_iter=10000,n_jobs=8)
    model.fit(Xtrain,y_train)
    Y_hat = model.decision_function(Xtest)
    test_aucs[i] = roc_auc_score(y_test,Y_hat)
    model_list.append(model)
    for j in range(9):
        test_aucs_mouse[i,j] = roc_auc_score(y_test[m_test==20+j],
                                np.squeeze(Y_hat[m_test==20+j]))
        U1, AUC1, U0, AUC0 = calc_U(y_test[m_test==20+j],np.squeeze(Y_hat[m_test==20+j]))
        test_aucs_mouse_U[i,j] = AUC1
    print(i,test_aucs_mouse[i])
    print(i,test_aucs_mouse_U[i])

In [ ]:
region_list = ['IL','LHb','LSN','MDThal','MeA','NAc','OFC','PL','V1','VHipp','VMHvl']
for i in range(11):
    print('Region ',region_list[i],test_aucs[i])

In [ ]:
import cloudpickle

In [ ]:
myDict = {'models':model_list}
with open('Exp8_2_nonAggressionVsAggressionFemaleCastrated_LinearModel.p','wb') as f:
    cloudpickle.dump(myDict,f)
np.savetxt('Exp8_2_nonAggressionVsAggressionFemaleCastrated_LinearModel.csv',
           test_aucs_mouse,fmt='%0.8f',delimiter=',')
np.savetxt('Exp8_2_nonAggressionVsAggressionFemaleCastrated_LinearModel_U.csv',
           test_aucs_mouse_U,fmt='%0.8f',delimiter=',')

In [ ]:
np.mean(test_aucs_mouse,axis=1)

In [ ]:
mouse_all2